
##Lesson 1: tracing
Why it matters
Every judge, every latency number and every token count in this project is read from a trace.


In [0]:
%pip install --upgrade "mlflow[databricks]>=3.4.0" openai datasets
dbutils.library.restartPython()

In [0]:
import mlflow
from mlflow.entities import Document
from databricks.sdk import WorkspaceClient

mlflow.openai.autolog()  # auto-traces every LLM call: tokens, latency
client = WorkspaceClient().serving_endpoints.get_open_ai_client()
MODEL = "databricks-gpt-oss-120b"  # replace with a chat model from your Serving page

DOCS = [
    Document(page_content="Delta Lake is an open table format that adds ACID transactions to data lakes.", metadata={"doc_uri": "d1"}),
    Document(page_content="MLflow Tracing records each step of a GenAI request as spans.", metadata={"doc_uri": "d2"}),
    Document(page_content="Hyderabad is the capital of Telangana.", metadata={"doc_uri": "d3"}),
]

@mlflow.trace(span_type="RETRIEVER")
def retrieve(question: str, k: int = 2):
    words = set(question.lower().split())
    return sorted(DOCS, key=lambda d: -len(words & set(d.page_content.lower().split())))[:k]

@mlflow.trace
def answer(question: str) -> str:
    docs = retrieve(question)
    context = "\n".join(d.page_content for d in docs)
    resp = client.chat.completions.create(model=MODEL, messages=[
        {"role": "system", "content": "Answer only from the context. If it isn't there, say you don't know."},
        {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}"},
    ])
    return resp.choices[0].message.content

answer("What does MLflow Tracing record?")

In [0]:
from datasets import load_dataset
ds = load_dataset("hotpotqa/hotpot_qa", "distractor", split="validation[:5]")
print(ds)

In [0]:
from databricks.sdk import WorkspaceClient
w = WorkspaceClient()
print(sorted(e.name for e in w.serving_endpoints.list() if e.name.startswith("databricks-")))

In [0]:
%sql
SELECT ai_classify(
  'Question: What is the capital of Telangana?
   Context: Hyderabad is the capital of Telangana.
   Answer: Hyderabad.',
  ARRAY('grounded', 'not_grounded')
) AS verdict

In [0]:
r = ds[0]
print(r)
print("Q:", r["question"]); print("A:", r["answer"], "|", r["type"], r["level"])
print("Supporting:", list(zip(r["supporting_facts"]["title"], r["supporting_facts"]["sent_id"])))
for t, sents in zip(r["context"]["title"], r["context"]["sentences"]):
    print(f"\n[{t}] {len(sents)} sentences"); print(" ", sents[0][:120])